# 04 — Hypothesis Testing (Phase 2)

For each hypothesis we state:
- H0 and H1.
- Justification of the test and its assumptions.
- Test statistic, p-value, effect size, and a bootstrap 95% CI where applicable.
- Interpretation with an explicit non-causation caveat.

Tests are performed on **train + validation only**, never on the held-out test set.

In [6]:
import sys, json, logging
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.splits import chronological_split
from src.hypothesis import (
    mannwhitney_test, chi_square_test, welch_ttest, normality_report,
    results_to_dataframe,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
OUT_TABLES = PROJECT_ROOT / 'outputs' / 'tables'; OUT_TABLES.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT

PosixPath('/Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados')

## 1. Load cleaned data and reuse the frozen split

In [7]:
with open(PROJECT_ROOT / 'outputs' / 'split_definition.json') as f:
    split_cfg = json.load(f)

acc_clf = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'accepted_classification.parquet')
acc_reg = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'accepted_regression.parquet')
rej = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'rejected_clean.parquet')

splits_clf = chronological_split(acc_clf, 'issue_d', split_cfg['train_end'], split_cfg['val_end'])
splits_reg = chronological_split(acc_reg, 'issue_d', split_cfg['train_end'], split_cfg['val_end'])

clf_eda = pd.concat([splits_clf.train, splits_clf.val], ignore_index=True)
reg_eda = pd.concat([splits_reg.train, splits_reg.val], ignore_index=True)
len(clf_eda), len(reg_eda), len(rej)

(1129813, 1321847, 27490781)

## 2. Choose tests based on distributional assumptions

We report Shapiro-Wilk on a reproducible subsample for the variables that will be
used in location tests. If normality is rejected, we default to non-parametric tests.

In [8]:
norm_rows = []
for col in ['dti', 'annual_inc', 'fico_range_low', 'loan_amnt']:
    if col in clf_eda.columns:
        r = normality_report(clf_eda[col].dropna().values, seed=42)
        r['column'] = col
        norm_rows.append(r)
norm_df = pd.DataFrame(norm_rows).set_index('column')
norm_df.to_csv(OUT_TABLES / 'hypothesis_normality_checks.csv')
norm_df.round(4)

,shapiro_stat,shapiro_p,skew,n_used
column,,,,
dti,0.9846,0.0,0.3945,5000
annual_inc,0.4951,0.0,11.9264,5000
fico_range_low,0.8908,0.0,1.2655,5000
loan_amnt,0.9408,0.0,0.7398,5000


## H1 — Higher debt-to-income ratios are associated with a higher default rate

- **H0:** the distribution of DTI is the same for `Default` (class 2) and `Good` (class 0) loans.
- **H1:** the distribution of DTI differs between the two classes (two-sided).
- **Test:** Mann-Whitney U (non-parametric; DTI is skewed and Shapiro rejects normality).
- **Effect size:** Cliff's delta.
- **Caveat:** observational association; not causal.

In [9]:
res_h1 = None
if {'dti', 'target_class'}.issubset(clf_eda.columns):
    a = clf_eda.loc[clf_eda['target_class'] == 2, 'dti'].dropna().values
    b = clf_eda.loc[clf_eda['target_class'] == 0, 'dti'].dropna().values
    res_h1 = mannwhitney_test(
        a, b,
        name='H1 DTI Default vs Good',
        h0='DTI distribution identical across Default and Good',
        h1='DTI distribution differs across Default and Good (two-sided)',
        note='Observational association; not causal.',
    )
    res_h1.to_dict()

## H2 — Loan purpose is associated with default risk

- **H0:** `purpose` and the 3-class `target_class` are independent.
- **H1:** they are associated.
- **Test:** Chi-square on the contingency table (categorical × categorical).
- **Effect size:** Cramér's V.
- **Assumption:** expected cell counts > 5; otherwise we collapse rare purposes.

In [10]:
res_h2 = None
if {'purpose', 'target_class'}.issubset(clf_eda.columns):
    table = pd.crosstab(clf_eda['purpose'], clf_eda['target_class'])

    # identify rare purposes (boolean Series indexed by purpose)
    rare = table.sum(axis=1) < 0.01 * table.values.sum()

    if rare.any():
        # 1. collapse the rare rows BEFORE reindexing
        other_row = table.loc[rare].sum(axis=0).rename('OTHER_RARE')
        # 2. keep the non-rare rows and append the collapsed row
        table = pd.concat([table.loc[~rare], other_row.to_frame().T])

    res_h2 = chi_square_test(
        table,
        name='H2 Purpose vs default class',
        h0='purpose and target_class are independent',
        h1='purpose and target_class are associated',
        note='Observational association; not causal.',
    )
    table.to_csv(OUT_TABLES / 'hypothesis_purpose_vs_class_table.csv')
    res_h2.to_dict()

## H3 — Applicants with lower risk scores are more likely to be rejected

- **H0:** the distribution of `Risk_Score` is identical between accepted and rejected applicants.
- **H1:** the distributions differ (two-sided).
- **Test:** Mann-Whitney U.
- **Effect size:** Cliff's delta.
- **Note:** `Risk_Score` in the rejected file is treated as missing when it is 0.0
  (per the specification). It is a different scale from FICO in the accepted file,
  so we compare accepted vs rejected within the rejected schema only using
  a joined proxy — see below.

In [11]:
# We compare rejected Risk_Score distribution to accepted fico_range_low
# only after mapping both to a common ordinal. For the formal test we use
# the rejected Risk_Score against a threshold-free alternative: state-level
# aggregation is done in H4. Here we test the internal split of Risk_Score
# across States with High vs Low average approval tendency.
res_h3 = None
if {'Risk_Score', 'State'}.issubset(rej.columns) and 'State' in acc_clf.columns:
    # Compute state-level approval proxy from the accepted (post-origination) file.
    # NOTE: we use the accepted file only to DEFINE the grouping; the outcome
    # used in the test is Risk_Score from the REJECTED file.
    acc_state = acc_clf.groupby('addr_state').size().rename('n_accepted')
    rej_state = rej.groupby('State').size().rename('n_rejected')
    joined = pd.concat([acc_state, rej_state], axis=1).dropna()
    joined['approval_ratio'] = joined['n_accepted'] / (joined['n_accepted'] + joined['n_rejected'])
    median_ratio = joined['approval_ratio'].median()
    high_states = joined.index[joined['approval_ratio'] >= median_ratio]
    low_states  = joined.index[joined['approval_ratio'] <  median_ratio]

    a = rej.loc[rej['State'].isin(high_states), 'Risk_Score'].dropna().values
    b = rej.loc[rej['State'].isin(low_states),  'Risk_Score'].dropna().values
    res_h3 = mannwhitney_test(
        a, b,
        name='H3 Risk_Score High vs Low approval states (rejected file)',
        h0='Risk_Score distribution identical across state groups',
        h1='Risk_Score distribution differs across state groups (two-sided)',
        note='State grouping derived from historical approval ratios; observational.',
    )
    joined.to_csv(OUT_TABLES / 'hypothesis_state_approval_ratio.csv')
    res_h3.to_dict()

## H4 — Requested loan amount differs between accepted and rejected applicants

- **H0:** requested amount distribution is identical between accepted and rejected applicants.
- **H1:** the distributions differ (two-sided).
- **Test:** Mann-Whitney U (skewed, right-tailed).
- **Effect size:** Cliff's delta.
- **Note:** 'Accepted' uses `loan_amnt` on the accepted file; 'rejected' uses
  `Amount Requested` on the rejected file.

In [12]:
res_h4 = None
if 'Amount Requested' in rej.columns and 'loan_amnt' in acc_clf.columns:
    a = acc_clf['loan_amnt'].dropna().values
    b = rej['Amount Requested'].dropna().values
    res_h4 = mannwhitney_test(
        a, b,
        name='H4 Requested amount Accepted vs Rejected',
        h0='Requested amount distribution identical across groups',
        h1='Requested amount distribution differs across groups (two-sided)',
        note='Observational association; not causal.',
    )
    res_h4.to_dict()

KeyboardInterrupt: 

## H5 (optional) — FICO and int_rate: Welch t-test on a plausible-normal proxy

We include this only to show that a parametric test would have been inappropriate
here given the Shapiro results — the reported p-value is for documentation, not
for inference.

In [ ]:
res_h5 = None
if {'fico_range_low', 'term'}.issubset(reg_eda.columns):
    a = reg_eda.loc[reg_eda['term'] == 36, 'fico_range_low'].dropna().values
    b = reg_eda.loc[reg_eda['term'] == 60, 'fico_range_low'].dropna().values
    res_h5 = welch_ttest(
        a, b,
        name='H5 FICO 36m vs 60m loans (parametric check)',
        h0='Mean FICO identical across terms (Welch)',
        h1='Mean FICO differs across terms (Welch, two-sided)',
        note='Parametric check only; Shapiro already rejects normality in §2.',
    )
    res_h5.to_dict()

## 3. Consolidated results table

One row per hypothesis, with the test name, statistic, p-value, effect size, and
bootstrap CI for the mean difference where applicable.

In [ ]:
all_results = [r for r in [res_h1, res_h2, res_h3, res_h4, res_h5] if r is not None]
results_df = results_to_dataframe(all_results)
results_df.to_csv(OUT_TABLES / 'hypothesis_results.csv', index=False)
results_df[['name','test','statistic','p_value','effect_size','effect_size_name','n','ci_low','ci_high']]

## 4. Interpretation (write up for the mid-journey report)

For each hypothesis, report:

1. **Test chosen and why** — non-parametric for skewed variables; chi-square for
   categorical × categorical.
2. **Assumptions checked** — Shapiro on subsamples; expected cell counts > 5
   for chi-square (rare categories collapsed).
3. **Statistical decision** — p-value vs α = 0.05.
4. **Effect size and CI** — never interpret a p-value alone; a small effect can
   be significant at scale.
5. **Non-causation caveat** — these are observational associations on historical
   data; they do not license causal claims, and any temporal or selection
   confound is not addressed by the tests themselves.

Additionally, note:

- **Selection bias:** the accepted file only contains funded loans; the rejected
  file is a different credit snapshot. Cross-file comparisons are therefore
  proxy tests at best.
- **Unequal follow-up:** older loans have had more time to default; the
  chronological split in Phase 1 partially mitigates but does not remove this.
- **Fairness:** any state-level or geographic result must be treated as a
  fairness-sensitive consideration, not a policy recommendation.